### Les contributions par lignes : 
- step 1 : on calcule par ligne 
- step 2 : on agrège

In [1]:
import pandas as pd

from railinsight.data.download import REGULARITE, load
from railinsight.data.quality import normaliser_noms

Chargement et nettoyage du nom des gares

In [2]:
df = load(REGULARITE)

for col in ["gare_depart", "gare_arrivee"]:
    df[col] = normaliser_noms(df[col])

print(df.shape)

(12907, 26)


Définition des 3 groupes de causes 

In [3]:
causes_reseau = ["prct_cause_infra", "prct_cause_gestion_trafic"]
causes_voyageurs = [
    "prct_cause_materiel_roulant",
    "prct_cause_gestion_gare",
    "prct_cause_prise_en_charge_voyageurs",
]
causes_externe = ["prct_cause_externe"]

Pour chaque ligne (liaison × mois), le pourcentage d’un groupe est son poids parmi les retards de cette ligne. Les trois pourcentages s’additionnent à 100.
Attention aux lignes dont la somme des causes est à 0 alors que des trains sont en retard (les ~80 faux zéros). 
Leur répartition est inconnue, il faudra les repérer et les mettre de côté, sinon leurs retards sont comptés sans cause

In [4]:
df["pct_reseau"] = df[causes_reseau].sum(axis=1)
df["pct_voyageurs"] = df[causes_voyageurs].sum(axis=1)
df["pct_externe"] = df[causes_externe].sum(axis=1)

df[["pct_reseau", "pct_voyageurs", "pct_externe"]].head()

,pct_reseau,pct_voyageurs,pct_externe
0,42.016807,21.848739,36.134454
1,51.666667,28.333333,20.000000
2,44.444444,33.333333,22.222222
3,38.888889,27.777778,33.333333
4,64.912281,19.298246,15.789474


Les trois colonnes de train en retard par groupe

In [5]:
df["nb_retard_reseau"] = df["nb_train_retard_arrivee"] * df["pct_reseau"] / 100
df["nb_retard_voyageurs"] = df["nb_train_retard_arrivee"] * df["pct_voyageurs"] / 100
df["nb_retard_externe"] = df["nb_train_retard_arrivee"] * df["pct_externe"] / 100

Le controle de cohérence

In [6]:
total_attribue = df["nb_retard_reseau"] + df["nb_retard_voyageurs"] + df["nb_retard_externe"]
ecart = total_attribue - df["nb_train_retard_arrivee"]
print(ecart.abs().describe())

count    1.290700e+04
mean     1.581312e-01
std      2.687682e+00
min      0.000000e+00
25%      0.000000e+00
50%      6.272707e-10
75%      2.400000e-09
max      1.160000e+02
dtype: float64


Count =	12 907	Toutes les lignes sont testées.  
min	= 0	Au mieux aucun écart.  
25%	= 0	Pour un quart des lignes, aucun écart.  
50%	= 6e-10	Pareil pour la moitié des lignes.  
75%	= 2,4e-9	Pareil pour 75% des lignes.  
max	= 116	Une ligne a 116 trains en retard non répartis.  
mean =	0,16	Moyenne, tirée vers le haut par les lignes problématiques.  
Pour la grande majorité des lignes, tout est bon : l’écart est de 0.
Quelques lignes ont un vrai problème. La pire en a 116 trains non répartis.

In [7]:
#Lignes ayant un vrai écart
print((ecart.abs() > 1).sum())

77


Cela représente 0,6%.
Vérifions ces 77 lignes

In [8]:
anomalies = ecart.abs() > 1

df.loc[anomalies, ["date", "gare_depart", "gare_arrivee", "nb_train_retard_arrivee", "pct_reseau", "pct_voyageurs", "pct_externe"]].head(10)

,date,gare_depart,gare_arrivee,nb_train_retard_arrivee,pct_reseau,pct_voyageurs,pct_externe
536,2018-05,BELLEGARDE (AIN),PARIS LYON,45,0.0,0.0,0.0
546,2018-05,PARIS LYON,ITALIE,13,0.0,0.0,0.0
547,2018-05,LAUSANNE,PARIS LYON,25,0.0,0.0,0.0
548,2018-05,PARIS LYON,ZURICH,35,0.0,0.0,0.0
571,2018-05,PARIS EST,FRANCFORT,11,0.0,0.0,0.0
572,2018-05,PARIS EST,STUTTGART,17,0.0,0.0,0.0
573,2018-05,MADRID,MARSEILLE ST CHARLES,3,0.0,0.0,0.0
600,2018-05,FRANCFORT,PARIS EST,38,0.0,0.0,0.0
601,2018-05,PARIS LYON,GENEVE,48,0.0,0.0,0.0
602,2018-05,GENEVE,PARIS LYON,43,0.0,0.0,0.0


In [9]:
df["cause_fiable"] = ~anomalies
print(df.loc[anomalies, "date"].value_counts().head(10))
print(df["cause_fiable"].value_counts())

date
2018-05    17
2018-07    16
2018-06    15
2018-08    15
2026-06     4
2020-06     3
2019-09     1
2019-11     1
2020-04     1
2026-05     1
Name: count, dtype: int64
cause_fiable
True     12830
False       77
Name: count, dtype: int64


In [10]:
gares_etrangeres = ["BARCELONA", "MADRID", "FRANCFORT", "GENEVE", "LAUSANNE", "STUTTGART", "ZURICH", "ITALIE"]
est_internationale = df["gare_depart"].isin(gares_etrangeres) | df["gare_arrivee"].isin(gares_etrangeres)
print(est_internationale[anomalies].mean())

0.8441558441558441


### Lignes sans cause renseignée

77 lignes (0,6 % du jeu) ont des causes à 0 % alors que des trains sont en retard.
- 63 d'entre elles datent de mai à août 2018.
- 84 % sont des liaisons internationales, alors que celles-ci ne représentent que 12 % des lignes du jeu (5,2 % des trains prévus).
Lignes conservées et marquées `cause_fiable = False`. Elles sont exclues du calcul des contributions par cause, mais restent dans les totaux de trains.

**Question pour le groupe** : on exclut les internationales ou on les marque juste ? 

### Agrégation par liaison
On additionne le nombre de trains et on divise.   
Gardons les lignes fiables

In [11]:
df_ok = df[df["cause_fiable"]]
print(df_ok.shape)

(12830, 33)


In [12]:
#La somme par liaison
par_liaison = df_ok.groupby(["gare_depart", "gare_arrivee"]).agg(
    nb_train_prevu=("nb_train_prevu", "sum"),
    nb_train_retard_arrivee=("nb_train_retard_arrivee", "sum"),
    nb_retard_reseau=("nb_retard_reseau", "sum"),
    nb_retard_voyageurs=("nb_retard_voyageurs", "sum"),
    nb_retard_externe=("nb_retard_externe", "sum"),
)
print(par_liaison.shape)
par_liaison.head()

(130, 5)


,,nb_train_prevu,nb_train_retard_arrivee,nb_retard_reseau,nb_retard_voyageurs,nb_retard_externe
gare_depart,gare_arrivee,,,,,
AIX EN PROVENCE TGV,PARIS LYON,43226,5131,2037.209110,1777.028002,1316.762888
ANGERS SAINT LAUD,PARIS MONTPARNASSE,50135,7433,3325.200530,2104.723428,2003.076043
ANGOULEME,PARIS MONTPARNASSE,29204,4670,2103.965403,1538.973391,1027.061206
ANNECY,PARIS LYON,14599,2837,1502.760529,522.940192,811.299280
ARRAS,PARIS NORD,43337,9134,5153.127405,2693.164921,1287.707674


In [13]:
par_liaison["contrib_reseau"] = par_liaison["nb_retard_reseau"] / par_liaison["nb_train_prevu"]
par_liaison["contrib_voyageurs"] = par_liaison["nb_retard_voyageurs"] / par_liaison["nb_train_prevu"]
par_liaison["contrib_externe"] = par_liaison["nb_retard_externe"] / par_liaison["nb_train_prevu"]

Contrôle

In [14]:
taux = par_liaison["nb_train_retard_arrivee"] / par_liaison["nb_train_prevu"]
somme_contrib = par_liaison["contrib_reseau"] + par_liaison["contrib_voyageurs"] + par_liaison["contrib_externe"]
print((somme_contrib - taux).abs().max())

0.003480278422273775


In [15]:
manque = (taux - somme_contrib) * par_liaison["nb_train_prevu"]
print(manque.sort_values(ascending=False).head(5))

gare_depart           gare_arrivee       
MARSEILLE ST CHARLES  MADRID                 3.0
LILLE                 LYON PART DIEU         2.0
PARIS MONTPARNASSE    ST PIERRE DES CORPS    1.0
PARIS LYON            MACON LOCHE            1.0
ITALIE                PARIS LYON             1.0
dtype: float64


### pour la datasheet :   
Écart résiduel après exclusion des 77 lignes : au plus 3 trains par liaison (0,35 point de taux sur la liaison la plus touchée, Marseille → Madrid, à faible volume)
sur le pire cas (Marseille - Madrid), 3 trains sur 860 ne sont pas rangés dans une cause, et sur les grosses liaisons c’est quasi nul

### Cause dominante
Pour chaque liaison, quelle cause de retard est la plus grande ?

In [16]:
colonnes_contrib = ["contrib_reseau", "contrib_voyageurs", "contrib_externe"]
par_liaison["cause_dominante"] = par_liaison[colonnes_contrib].idxmax(axis=1)
print(par_liaison["cause_dominante"].value_counts())
print(par_liaison["cause_dominante"].value_counts(normalize=True).round(2))

cause_dominante
contrib_reseau       83
contrib_voyageurs    44
contrib_externe       3
Name: count, dtype: int64
cause_dominante
contrib_reseau       0.64
contrib_voyageurs    0.34
contrib_externe      0.02
Name: proportion, dtype: float64


Pour 64% des liaisons, la cause dominante est le réseau (infra + trafic)
34% = les voyageurs
2% externe

### Test de robustesse du cadrage
Etape 1 = nbre de train en retard dus à la prise en charge voyageur

In [17]:
df_ok2 = df_ok.assign(
    nb_retard_pec=df_ok["nb_train_retard_arrivee"] * df_ok["prct_cause_prise_en_charge_voyageurs"] / 100
)
par_liaison["nb_retard_pec"] = df_ok2.groupby(["gare_depart", "gare_arrivee"])["nb_retard_pec"].sum()
par_liaison[["nb_retard_voyageurs", "nb_retard_pec"]].head()

,,nb_retard_voyageurs,nb_retard_pec
gare_depart,gare_arrivee,,
AIX EN PROVENCE TGV,PARIS LYON,1777.028002,437.947840
ANGERS SAINT LAUD,PARIS MONTPARNASSE,2104.723428,624.527308
ANGOULEME,PARIS MONTPARNASSE,1538.973391,576.715945
ANNECY,PARIS LYON,522.940192,89.719928
ARRAS,PARIS NORD,2693.164921,425.341032


In [18]:
print((par_liaison["nb_retard_pec"] <= par_liaison["nb_retard_voyageurs"] + 1e-9).all())

True


Variante A = les problèmes de prises en charge passent dans cause Externe

In [19]:
a_reseau = par_liaison["nb_retard_reseau"]
a_voyageurs = par_liaison["nb_retard_voyageurs"] - par_liaison["nb_retard_pec"]
a_externe = par_liaison["nb_retard_externe"] + par_liaison["nb_retard_pec"]

variante_a = pd.concat([a_reseau, a_voyageurs, a_externe], axis=1, keys=["reseau", "voyageurs", "externe"])
print(variante_a.idxmax(axis=1).value_counts())

reseau       92
voyageurs    24
externe      14
Name: count, dtype: int64


Variante B : les problèmes de prises en charge passent dans cause Réseau

In [20]:
b_reseau = par_liaison["nb_retard_reseau"] + par_liaison["nb_retard_pec"]
b_voyageurs = par_liaison["nb_retard_voyageurs"] - par_liaison["nb_retard_pec"]
b_externe = par_liaison["nb_retard_externe"]

variante_b = pd.concat([b_reseau, b_voyageurs, b_externe], axis=1, keys=["reseau", "voyageurs", "externe"])
print(variante_b.idxmax(axis=1).value_counts())

reseau       116
voyageurs     12
externe        2
Name: count, dtype: int64


### Conclusion du test de robustesse
Le classement des causes dominantes ne change pas : Réseau > Voyageurs > Externe, quelle que soit la nouvelle cause dans laquelle on met "prise en charge voyageurs".
- Le Réseau domine dans 64 % des liaisons (cadrage), 71 % (variante A) et 89 % (variante B).
- La part des Voyageurs est la plus sensible : 44 liaisons dans le cadrage, 24 ou 12 selon la variante.
- Limite : dans les deux variantes, le Réseau ne perd aucune cause, donc le test ne pouvait pas le faire passer derrière. 

### Mise en place test plus sévère

Test sévère : sortir la gestion du trafic du Réseau

Dans les variantes A et B, le Réseau ne perd aucune cause : le test ne pouvait donc pas le faire passer derrière. Ici, on lui retire l'une de ses deux causes, `gestion_trafic` (l'autre étant `infra`), pour voir si sa première place tient.

- **Étape 1** : calculer le nombre de trains en retard dus à la gestion du trafic, par liaison, et vérifier qu'il ne dépasse jamais le total Réseau.
- **Variante C** : la gestion du trafic passe chez les Voyageurs.
- **Variante D** : la gestion du trafic passe chez l'Externe.

Dans chaque variante, on retire un groupe pour ajouter à un autre, donc la somme des trois groupes ne doit pas changer. On vérifie aussi qu'aucune valeur ne devient négative.

In [21]:
df_ok3 = df_ok.assign(
    nb_retard_gestion_trafic=df_ok["nb_train_retard_arrivee"] * df_ok["prct_cause_gestion_trafic"] / 100
)
par_liaison["nb_retard_gestion_trafic"] = df_ok3.groupby(["gare_depart", "gare_arrivee"])["nb_retard_gestion_trafic"].sum()
print((par_liaison["nb_retard_gestion_trafic"] <= par_liaison["nb_retard_reseau"] + 1e-9).all())

True


In [22]:
c_reseau = par_liaison["nb_retard_reseau"] - par_liaison["nb_retard_gestion_trafic"]
c_voyageurs = par_liaison["nb_retard_voyageurs"] + par_liaison["nb_retard_gestion_trafic"]
c_externe = par_liaison["nb_retard_externe"]

variante_c = pd.concat([c_reseau, c_voyageurs, c_externe], axis=1, keys=["reseau", "voyageurs", "externe"])
print(variante_c.idxmax(axis=1).value_counts())

d_reseau = par_liaison["nb_retard_reseau"] - par_liaison["nb_retard_gestion_trafic"]
d_voyageurs = par_liaison["nb_retard_voyageurs"] 
d_externe = par_liaison["nb_retard_externe"] + par_liaison["nb_retard_gestion_trafic"]

variante_d = pd.concat([d_reseau, d_voyageurs, d_externe], axis=1, keys=["reseau", "voyageurs", "externe"])
print(variante_d.idxmax(axis=1).value_counts())

voyageurs    129
externe        1
Name: count, dtype: int64
externe      87
voyageurs    43
Name: count, dtype: int64


In [23]:
print((variante_c < 0).any().any(), (variante_d < 0).any().any())

depart = par_liaison["nb_retard_reseau"] + par_liaison["nb_retard_voyageurs"] + par_liaison["nb_retard_externe"]
print((variante_c.sum(axis=1) - depart).abs().max())
print((variante_d.sum(axis=1) - depart).abs().max())

False False
1.8189894035458565e-12
1.8189894035458565e-12


In [24]:
infra = par_liaison["nb_retard_reseau"] - par_liaison["nb_retard_gestion_trafic"]
print("infra :", round(infra.sum()))
print("gestion trafic :", round(par_liaison["nb_retard_gestion_trafic"].sum()))
print("voyageurs :", round(par_liaison["nb_retard_voyageurs"].sum()))
print("externe :", round(par_liaison["nb_retard_externe"].sum()))

infra : 102683
gestion trafic : 103872
voyageurs : 181609
externe : 111227


**Résultat** : dans les deux variantes, le Réseau n'est plus la cause dominante d'aucune liaison.

| Version | Réseau | Voyageurs | Externe |
|---|---|---|---|
| C (gestion du trafic → Voyageurs) | 0 | 129 | 1 |
| D (gestion du trafic → Externe) | 0 | 43 | 87 |

La domination du Réseau repose donc sur ses deux causes réunies : infra et gestion du trafic pèsent presque autant l'une que l'autre (≈ 21 % des trains en retard chacune), et aucune des deux n'est dominante seule.

Ce test est volontairement extrême : on déplace toute la gestion du trafic vers un seul groupe, qui additionne alors plusieurs causes. Il montre la dépendance au regroupement, pas qu'une autre famille domine réellement.

### Conclusion de la Phase 1
- Regroupement retenu (cadrage) : Réseau (infra, gestion du trafic), Voyageurs (matériel roulant, gestion de gare, prise en charge), Externe.
- Cause dominante par liaison : Réseau 64 %, Voyageurs 34 %, Externe 2 %.
- Impact du regroupement : la domination du Réseau repose sur deux causes de poids égal (infra et gestion du trafic, ~21 % chacune des retards). Si la gestion du trafic est classée hors du Réseau, le Réseau ne domine plus aucune liaison.
- Le classement de la prise en charge change peu la conclusion (Réseau dominant dans 64 à 89 % des liaisons), mais fait beaucoup varier la part des Voyageurs (44 → 12 à 24 liaisons).
- Ordre de grandeur (exploratoire) : Réseau ~41 %, Voyageurs ~36 %, Externe ~22 %. Les trois familles sont proches.

### pour la datasheet
Les causes sont regroupées en trois familles selon le cadrage.
Un regroupement de plusieurs causes avantage mécaniquement les familles les plus larges (Voyageurs : 3 causes, Réseau : 2, Externe : 1).
Les résultats sont sensibles au classement de gestion_trafic et de prise_en_charge_voyageurs.